In [1]:
from pathlib import Path

import polars as pl

DATA_PATH = Path("processed_reviews.parquet")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}"
    )

reviews = pl.scan_parquet(DATA_PATH)

print("Schema:")
print(reviews.collect_schema())

print("\nSample rows:")
display(reviews.head(5).collect())

Schema:
Schema({'recommendationid': Int64, 'appid': Int64, 'game': String, 'author_steamid': Int64, 'author_num_games_owned': Int64, 'author_num_reviews': Int64, 'author_playtime_forever': Float64, 'author_playtime_last_two_weeks': Float64, 'author_playtime_at_review': Int64, 'author_last_played': Float64, 'language': String, 'review': String, 'timestamp_created': Int64, 'timestamp_updated': Int64, 'voted_up': Int64, 'votes_up': Int64, 'votes_funny': Int64, 'weighted_vote_score': Float64, 'comment_count': Int64, 'steam_purchase': Int64, 'received_for_free': Int64, 'written_during_early_access': Int64, 'hidden_in_steam_china': Int64, 'steam_china_location': String})

Sample rows:


recommendationid,appid,game,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,hidden_in_steam_china,steam_china_location
i64,i64,str,i64,i64,i64,f64,f64,i64,f64,str,str,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,str
148919893,10,"""Counter-Strike""",76561199036724879,0,3,197.0,197.0,197,1.6983e9,"""russian""","""старость""",1698336397,1698336397,1,0,0,0.0,0,1,0,0,1,null
148919350,10,"""Counter-Strike""",76561198826729322,0,21,441.0,37.0,441,1.6983e9,"""russian""","""Лучше кс 2""",1698335821,1698335821,1,0,0,0.0,0,1,0,0,1,null
148912714,10,"""Counter-Strike""",76561198347582422,11,4,1636.0,83.0,1612,1.6983e9,"""russian""","""топ""",1698329555,1698329555,1,0,0,0.0,0,1,0,0,1,null
148905699,10,"""Counter-Strike""",76561198820122182,88,39,11.0,0.0,11,1.6958e9,"""turkish""","""eskisi kadar sarmiyor""",1698321259,1698321259,1,0,0,0.0,0,0,0,0,1,null
148899121,10,"""Counter-Strike""",76561199143791757,9,4,1271.0,973.0,1202,1.6983e9,"""russian""","""топ""",1698312060,1698312060,1,0,0,0.0,0,1,0,0,1,null


In [5]:
USER_COLUMN = "author_steamid"
GAME_COLUMN = "appid"
LABEL_COLUMN = "voted_up"



In [8]:
# ============================================================
# STEP 2: Create stable integer indexes for users and games
# ============================================================

user_index = (
    reviews
    .select("author_steamid")
    .unique()
    .sort("author_steamid")
    .with_row_index("user_idx")
    .collect()
)

game_index = (
    reviews
    .select("appid", "game")
    .unique(subset=["appid"])
    .sort("appid")
    .with_row_index("game_idx")
    .collect()
)

print("Users:", user_index.height)
print("Games:", game_index.height)

display(user_index.head())
display(game_index.head())

Users: 10495262
Games: 74815


user_idx,author_steamid
u32,i64
0,76561197960265745
1,76561197960265778
2,76561197960265781
3,76561197960265822
4,76561197960265841


game_idx,appid,game
u32,i64,str
0,10,"""Counter-Strike"""
1,20,"""Team Fortress Classic"""
2,30,"""Day of Defeat"""
3,40,"""Deathmatch Classic"""
4,50,"""Half-Life: Opposing Force"""


In [10]:
# ============================================================
# STEP 3: Join integer user/game indexes onto each review
# Join games using appid only, since game names may vary
# ============================================================

interactions = (
    reviews
    .join(
        user_index.lazy(),
        on="author_steamid",
        how="inner",
    )
    .join(
        game_index.select("appid", "game_idx").lazy(),
        on="appid",
        how="inner",
    )
    .select(
        "user_idx",
        "game_idx",
        "voted_up",
        "timestamp_created",
    )
)

In [11]:
# ============================================================
# STEP 4: Verify that the joins preserved every review row
# ============================================================

original_row_count = reviews.select(
    pl.len().alias("rows")
).collect()["rows"][0]

interaction_row_count = interactions.select(
    pl.len().alias("rows")
).collect()["rows"][0]

print("Original rows:   ", original_row_count)
print("Interaction rows:", interaction_row_count)
print("Rows match:      ", original_row_count == interaction_row_count)

Original rows:    79844333
Interaction rows: 79844333
Rows match:       True


In [12]:
# ============================================================
# STEP 5: Save the model-ready interaction table
# ============================================================

interactions.sink_parquet(
    "model_interactions.parquet",
    compression="zstd",
)

print("Saved model_interactions.parquet")

Saved model_interactions.parquet


In [18]:
# ============================================================
# STEP 6: Create a random one-review holdout for each user
# This avoids biasing the test set toward newer games.
# ============================================================

model_interactions = pl.scan_parquet("model_interactions.parquet")

split_interactions = (
    model_interactions
    # Create a reproducible random value for each review row
    .with_columns(
        pl.int_range(pl.len())
        .shuffle(seed=42)
        .over("user_idx")
        .alias("random_rank")
    )
)

# Hold out one random review per user
test_interactions = (
    split_interactions
    .filter(pl.col("random_rank") == 0)
    .drop("random_rank")
)

# Keep all remaining reviews for training
train_interactions = (
    split_interactions
    .filter(pl.col("random_rank") > 0)
    .drop("random_rank")
)

print("Random per-user split definitions created.")

Random per-user split definitions created.


In [19]:
# ============================================================
# STEP 7: Materialize the lazy train/test splits to Parquet
# ============================================================

train_interactions.sink_parquet(
    "train_interactions.parquet",
    compression="zstd",
)

test_interactions.sink_parquet(
    "test_interactions.parquet",
    compression="zstd",
)

print("Saved train_interactions.parquet")
print("Saved test_interactions.parquet")

Saved train_interactions.parquet
Saved test_interactions.parquet


In [20]:
# ============================================================
# STEP 8: Verify the train/test split
# Each user should contribute exactly one test review
# ============================================================

train_reviews = pl.scan_parquet("train_interactions.parquet")
test_reviews = pl.scan_parquet("test_interactions.parquet")

split_summary = pl.DataFrame(
    {
        "dataset": ["train", "test"],
        "rows": [
            train_reviews.select(pl.len()).collect().item(),
            test_reviews.select(pl.len()).collect().item(),
        ],
        "users": [
            train_reviews.select(pl.col("user_idx").n_unique()).collect().item(),
            test_reviews.select(pl.col("user_idx").n_unique()).collect().item(),
        ],
    }
)

display(split_summary)

dataset,rows,users
str,i64,i64
"""train""",69349071,10495262
"""test""",10495262,10495262


In [21]:
# ============================================================
# STEP 9: Verify the model label values
# ============================================================

label_values = (
    train_reviews
    .select("voted_up")
    .unique()
    .sort("voted_up")
    .collect()
)

display(label_values)

voted_up
i64
0
1


In [22]:
# ============================================================
# STEP 10: Create a streaming PyTorch dataset
# Reads the Parquet file in batches during training
# ============================================================

import pyarrow.parquet as pq
import torch
from torch.utils.data import IterableDataset


class ParquetReviewDataset(IterableDataset):
    def __init__(
        self,
        parquet_path: str,
        batch_size: int = 65_536,
    ):
        self.parquet_path = parquet_path
        self.batch_size = batch_size

    def __iter__(self):
        parquet_file = pq.ParquetFile(self.parquet_path)

        for batch in parquet_file.iter_batches(
            batch_size=self.batch_size,
            columns=["user_idx", "game_idx", "voted_up"],
        ):
            batch_dict = batch.to_pydict()

            user_indices = torch.tensor(
                batch_dict["user_idx"],
                dtype=torch.long,
            )

            game_indices = torch.tensor(
                batch_dict["game_idx"],
                dtype=torch.long,
            )

            labels = torch.tensor(
                batch_dict["voted_up"],
                dtype=torch.float32,
            )

            yield user_indices, game_indices, labels


train_dataset = ParquetReviewDataset(
    "train_interactions.parquet"
)

test_dataset = ParquetReviewDataset(
    "test_interactions.parquet"
)

print("Streaming datasets created.")

Streaming datasets created.


In [23]:
# ============================================================
# STEP 11: Define the user/game embedding model
# ============================================================

import torch
from torch import nn


class MatrixFactorizationModel(nn.Module):
    def __init__(
        self,
        num_users: int,
        num_games: int,
        embedding_dim: int = 64,
    ):
        super().__init__()

        # One learned vector for every historical Steam user
        self.user_embeddings = nn.Embedding(
            num_embeddings=num_users,
            embedding_dim=embedding_dim,
        )

        # One learned vector for every game
        self.game_embeddings = nn.Embedding(
            num_embeddings=num_games,
            embedding_dim=embedding_dim,
        )

        # These capture general tendencies:
        # - Some users review almost everything positively
        # - Some games receive positive reviews from almost everyone
        self.user_biases = nn.Embedding(num_users, 1)
        self.game_biases = nn.Embedding(num_games, 1)

        # Overall baseline positivity across the dataset
        self.global_bias = nn.Parameter(torch.zeros(1))

        # Start embeddings with small random values
        nn.init.normal_(self.user_embeddings.weight, std=0.05)
        nn.init.normal_(self.game_embeddings.weight, std=0.05)

        # Start bias values at zero
        nn.init.zeros_(self.user_biases.weight)
        nn.init.zeros_(self.game_biases.weight)

    def forward(
        self,
        user_idx: torch.Tensor,
        game_idx: torch.Tensor,
    ) -> torch.Tensor:

        # Retrieve the vectors for each user and game in the batch
        user_vectors = self.user_embeddings(user_idx)
        game_vectors = self.game_embeddings(game_idx)

        # Measure user-game compatibility with a dot product
        compatibility = (user_vectors * game_vectors).sum(dim=1)

        # Retrieve user-specific and game-specific bias values
        user_bias = self.user_biases(user_idx).squeeze(dim=1)
        game_bias = self.game_biases(game_idx).squeeze(dim=1)

        # Return raw prediction scores, called logits
        logits = (
            compatibility
            + user_bias
            + game_bias
            + self.global_bias
        )

        return logits


print("MatrixFactorizationModel defined.")

MatrixFactorizationModel defined.


In [24]:
# ============================================================
# STEP 12: Create the model, loss function, and optimizer
# ============================================================

# Number of unique users and games from the index tables
NUM_USERS = user_index.height
NUM_GAMES = game_index.height

# Use the GPU when available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Create the model
model = MatrixFactorizationModel(
    num_users=NUM_USERS,
    num_games=NUM_GAMES,
    embedding_dim=64,
).to(device)

# Binary classification loss:
# predicts positive review (1) versus negative review (0)
loss_function = nn.BCEWithLogitsLoss()

# Optimizer updates the learned vectors and biases
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.00001,
)

print("Device:", device)
print("Users:", NUM_USERS)
print("Games:", NUM_GAMES)

Device: cuda
Users: 10495262
Games: 74815


In [25]:
# ============================================================
# STEP 13: Create training and test data loaders
# ============================================================

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=None,   # Dataset already yields complete batches
    num_workers=0,     # Keep this at 0 for the first run
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=None,
    num_workers=0,
    pin_memory=True,
)

print("Training and test data loaders created.")

Training and test data loaders created.


In [26]:
# ============================================================
# STEP 14: Train the model for one epoch
# ============================================================

model.train()

total_loss = 0.0
total_rows = 0

for batch_number, (user_idx, game_idx, labels) in enumerate(train_loader, start=1):

    # Move the current batch from CPU memory to the GPU
    user_idx = user_idx.to(device, non_blocking=True)
    game_idx = game_idx.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    # Clear gradients left over from the previous batch
    optimizer.zero_grad(set_to_none=True)

    # Predict whether each user-game review is positive
    logits = model(user_idx, game_idx)

    # Compare predictions against the actual review labels
    loss = loss_function(logits, labels)

    # Calculate gradients and update embeddings/biases
    loss.backward()
    optimizer.step()

    # Track the average loss across all processed rows
    batch_rows = labels.size(0)
    total_loss += loss.item() * batch_rows
    total_rows += batch_rows

    # Print occasional progress
    if batch_number % 100 == 0:
        average_loss = total_loss / total_rows

        print(
            f"Batch {batch_number:,} | "
            f"Rows {total_rows:,} | "
            f"Average loss {average_loss:.5f}"
        )

epoch_loss = total_loss / total_rows

print()
print(f"Epoch complete")
print(f"Training rows: {total_rows:,}")
print(f"Average training loss: {epoch_loss:.5f}")

Batch 100 | Rows 6,553,600 | Average loss 0.67326
Batch 200 | Rows 13,107,200 | Average loss 0.65615
Batch 300 | Rows 19,660,800 | Average loss 0.64534
Batch 400 | Rows 26,214,400 | Average loss 0.63011
Batch 500 | Rows 32,768,000 | Average loss 0.61616
Batch 600 | Rows 39,321,600 | Average loss 0.60374
Batch 700 | Rows 45,875,200 | Average loss 0.59291
Batch 800 | Rows 52,428,800 | Average loss 0.58010
Batch 900 | Rows 58,982,400 | Average loss 0.57124
Batch 1,000 | Rows 65,536,000 | Average loss 0.56214

Epoch complete
Training rows: 69,349,071
Average training loss: 0.55718


In [31]:
# ============================================================
# STEP 15: Evaluate the model on the held-out test reviews
# ============================================================

model.eval()

test_loss_total = 0.0
test_correct = 0
test_rows = 0

# Disable gradient calculation because we are not training
with torch.no_grad():

    for user_idx, game_idx, labels in test_loader:

        # Move the batch to the GPU
        user_idx = user_idx.to(device, non_blocking=True)
        game_idx = game_idx.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Generate predictions
        logits = model(user_idx, game_idx)

        # Calculate test loss
        loss = loss_function(logits, labels)

        # Convert raw logits into probabilities
        probabilities = torch.sigmoid(logits)

        # Classify probability >= 0.5 as a positive review
        predictions = probabilities >= 0.5

        # Accumulate metrics
        batch_rows = labels.size(0)

        test_loss_total += loss.item() * batch_rows
        test_correct += (
            predictions == labels.bool()
        ).sum().item()

        test_rows += batch_rows


average_test_loss = test_loss_total / test_rows
test_accuracy = test_correct / test_rows

print(f"Test rows:     {test_rows:,}")
print(f"Test loss:     {average_test_loss:.5f}")
print(f"Test accuracy: {test_accuracy:.4%}")

Test rows:     10,495,262
Test loss:     0.35421
Test accuracy: 86.7749%


In [32]:
# ============================================================
# STEP 16: Compare the model against a constant baseline
# Baseline always predicts the test set's overall positive rate
# ============================================================

import math

test_positive_rate = (
    test_reviews
    .select(pl.col("voted_up").mean())
    .collect()
    .item()
)

# Accuracy from predicting every review as positive
baseline_accuracy = test_positive_rate

# Binary cross-entropy from predicting the same probability
# for every test review
baseline_log_loss = -(
    test_positive_rate * math.log(test_positive_rate)
    + (1 - test_positive_rate) * math.log(1 - test_positive_rate)
)

print(f"Test positive rate:     {test_positive_rate:.4%}")
print(f"Baseline accuracy:      {baseline_accuracy:.4%}")
print(f"Model accuracy:         {test_accuracy:.4%}")
print(f"Baseline log loss:      {baseline_log_loss:.5f}")
print(f"Model test loss:        {average_test_loss:.5f}")

Test positive rate:     86.8865%
Baseline accuracy:      86.8865%
Model accuracy:         86.7749%
Baseline log loss:      0.38854
Model test loss:        0.35421


In [33]:
# ============================================================
# STEP 17: Inspect the model's prediction distribution
# This checks whether it predicts nearly everything as positive.
# ============================================================

model.eval()

total_rows = 0
total_predicted_positive = 0
total_probability = 0.0

with torch.no_grad():

    for user_idx, game_idx, labels in test_loader:

        user_idx = user_idx.to(device, non_blocking=True)
        game_idx = game_idx.to(device, non_blocking=True)

        logits = model(user_idx, game_idx)
        probabilities = torch.sigmoid(logits)

        total_rows += probabilities.numel()

        total_predicted_positive += (
            probabilities >= 0.5
        ).sum().item()

        total_probability += probabilities.sum().item()


predicted_positive_rate = (
    total_predicted_positive / total_rows
)

average_predicted_probability = (
    total_probability / total_rows
)

print(f"Predicted positive rate:       {predicted_positive_rate:.4%}")
print(f"Average predicted probability: {average_predicted_probability:.4%}")
print(f"Actual positive rate:          {test_positive_rate:.4%}")

Predicted positive rate:       99.3219%
Average predicted probability: 85.7673%
Actual positive rate:          86.8865%


In [30]:
# ============================================================
# STEP 18: Train for three additional epochs
# ============================================================

def train_one_epoch(
    model,
    train_loader,
    optimizer,
    loss_function,
    device,
    epoch_number,
):
    model.train()

    total_loss = 0.0
    total_rows = 0

    for batch_number, (user_idx, game_idx, labels) in enumerate(
        train_loader,
        start=1,
    ):
        user_idx = user_idx.to(device, non_blocking=True)
        game_idx = game_idx.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(user_idx, game_idx)
        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()

        batch_rows = labels.size(0)
        total_loss += loss.item() * batch_rows
        total_rows += batch_rows

        if batch_number % 250 == 0:
            average_loss = total_loss / total_rows

            print(
                f"Epoch {epoch_number} | "
                f"Batch {batch_number:,} | "
                f"Average loss {average_loss:.5f}"
            )

    epoch_loss = total_loss / total_rows

    print(
        f"Epoch {epoch_number} complete | "
        f"Average training loss: {epoch_loss:.5f}"
    )

    return epoch_loss


for epoch_number in range(2, 5):
    train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        loss_function=loss_function,
        device=device,
        epoch_number=epoch_number,
    )

Epoch 2 | Batch 250 | Average loss 0.44861
Epoch 2 | Batch 500 | Average loss 0.43899
Epoch 2 | Batch 750 | Average loss 0.42704
Epoch 2 | Batch 1,000 | Average loss 0.42124
Epoch 2 complete | Average training loss: 0.42004
Epoch 3 | Batch 250 | Average loss 0.37920
Epoch 3 | Batch 500 | Average loss 0.37594
Epoch 3 | Batch 750 | Average loss 0.36698
Epoch 3 | Batch 1,000 | Average loss 0.36335
Epoch 3 complete | Average training loss: 0.36324
Epoch 4 | Batch 250 | Average loss 0.32725
Epoch 4 | Batch 500 | Average loss 0.32638
Epoch 4 | Batch 750 | Average loss 0.31823
Epoch 4 | Batch 1,000 | Average loss 0.31440
Epoch 4 complete | Average training loss: 0.31505


In [34]:
# ============================================================
# STEP 19: Compare scores for positive and negative reviews
# ============================================================

model.eval()

positive_probability_sum = 0.0
negative_probability_sum = 0.0

positive_rows = 0
negative_rows = 0

with torch.no_grad():

    for user_idx, game_idx, labels in test_loader:

        user_idx = user_idx.to(device, non_blocking=True)
        game_idx = game_idx.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(user_idx, game_idx)
        probabilities = torch.sigmoid(logits)

        positive_mask = labels == 1
        negative_mask = labels == 0

        positive_probability_sum += (
            probabilities[positive_mask].sum().item()
        )

        negative_probability_sum += (
            probabilities[negative_mask].sum().item()
        )

        positive_rows += positive_mask.sum().item()
        negative_rows += negative_mask.sum().item()


average_positive_probability = (
    positive_probability_sum / positive_rows
)

average_negative_probability = (
    negative_probability_sum / negative_rows
)

print(f"Positive review rows:        {positive_rows:,}")
print(f"Negative review rows:        {negative_rows:,}")
print(
    f"Average score for positives: "
    f"{average_positive_probability:.4%}"
)
print(
    f"Average score for negatives: "
    f"{average_negative_probability:.4%}"
)
print(
    f"Average score separation:    "
    f"{average_positive_probability - average_negative_probability:.4%}"
)

Positive review rows:        9,118,964
Negative review rows:        1,376,298
Average score for positives: 86.7667%
Average score for negatives: 79.1457%
Average score separation:    7.6210%


In [36]:
# ============================================================
# STEP 20: Calculate ROC AUC on held-out reviews
# AUC measures how often a positive review receives a higher
# score than a negative review.
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score

model.eval()

all_probabilities = []
all_labels = []

with torch.no_grad():

    for user_idx, game_idx, labels in test_loader:

        user_idx = user_idx.to(device, non_blocking=True)
        game_idx = game_idx.to(device, non_blocking=True)

        logits = model(user_idx, game_idx)
        probabilities = torch.sigmoid(logits)

        # Move results back to CPU for sklearn
        all_probabilities.append(
            probabilities.cpu().numpy().astype(np.float32)
        )

        all_labels.append(
            labels.numpy().astype(np.uint8)
        )


all_probabilities = np.concatenate(all_probabilities)
all_labels = np.concatenate(all_labels)

test_auc = roc_auc_score(
    all_labels,
    all_probabilities,
)

print(f"Test ROC AUC: {test_auc:.4f}")

Test ROC AUC: 0.7466


In [37]:
# ============================================================
# STEP 21: Save the trained model and index mappings
# ============================================================

MODEL_PATH = "matrix_factorization_model.pt"
USER_INDEX_PATH = "user_index.parquet"
GAME_INDEX_PATH = "game_index.parquet"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "num_users": NUM_USERS,
        "num_games": NUM_GAMES,
        "embedding_dim": 64,
        "epochs_trained": 4,
        "test_loss": average_test_loss,
        "test_auc": test_auc,
    },
    MODEL_PATH,
)

user_index.write_parquet(
    USER_INDEX_PATH,
    compression="zstd",
)

game_index.write_parquet(
    GAME_INDEX_PATH,
    compression="zstd",
)

print(f"Saved {MODEL_PATH}")
print(f"Saved {USER_INDEX_PATH}")
print(f"Saved {GAME_INDEX_PATH}")

Saved matrix_factorization_model.pt
Saved user_index.parquet
Saved game_index.parquet


In [39]:
# ============================================================
# STEP 23: Restore epoch 4 and recreate the best epoch 5 model
# ============================================================

checkpoint = torch.load(
    "matrix_factorization_model.pt",
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

train_loss_epoch_5 = train_one_epoch(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    loss_function=loss_function,
    device=device,
    epoch_number=5,
)

test_loss_epoch_5, test_auc_epoch_5 = evaluate_model(
    model=model,
    test_loader=test_loader,
    loss_function=loss_function,
    device=device,
)

print(f"Epoch 5 test loss: {test_loss_epoch_5:.5f}")
print(f"Epoch 5 test AUC:  {test_auc_epoch_5:.4f}")

Epoch 5 | Batch 250 | Average loss 0.27503
Epoch 5 | Batch 500 | Average loss 0.27658
Epoch 5 | Batch 750 | Average loss 0.27040
Epoch 5 | Batch 1,000 | Average loss 0.26766
Epoch 5 complete | Average training loss: 0.26903
Epoch 5 test loss: 0.35173
Epoch 5 test AUC:  0.7488


In [40]:
# ============================================================
# STEP 24: Save the selected epoch 5 model
# ============================================================

BEST_MODEL_PATH = "matrix_factorization_epoch5.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "num_users": NUM_USERS,
        "num_games": NUM_GAMES,
        "embedding_dim": 64,
        "epochs_trained": 5,
        "test_loss": test_loss_epoch_5,
        "test_auc": test_auc_epoch_5,
    },
    BEST_MODEL_PATH,
)

print(f"Saved {BEST_MODEL_PATH}")
print(f"Test loss: {test_loss_epoch_5:.5f}")
print(f"Test AUC:  {test_auc_epoch_5:.4f}")

Saved matrix_factorization_epoch5.pt
Test loss: 0.35173
Test AUC:  0.7488


In [41]:
# ============================================================
# STEP 25: Create a positive held-out evaluation sample
# ============================================================

EVALUATION_USERS = 100_000
NEGATIVE_CANDIDATES = 100
RANDOM_SEED = 42

positive_test_sample = (
    pl.scan_parquet("test_interactions.parquet")
    .filter(pl.col("voted_up") == 1)
    .select(
        "user_idx",
        pl.col("game_idx").alias("target_game_idx"),
    )
    .collect()
    .sample(
        n=EVALUATION_USERS,
        seed=RANDOM_SEED,
        shuffle=True,
    )
)

print(f"Evaluation users: {positive_test_sample.height:,}")
display(positive_test_sample.head())

Evaluation users: 100,000


user_idx,target_game_idx
u32,u32
4084903,29
8142465,10473
4781223,32816
9014067,29
778724,24726


In [42]:
# ============================================================
# STEP 26: Generate random candidate games for each user
# ============================================================

import numpy as np

rng = np.random.default_rng(RANDOM_SEED)

target_game_indices = (
    positive_test_sample
    .get_column("target_game_idx")
    .to_numpy()
)

# Shape:
# 100,000 users × 100 random comparison games
random_candidate_indices = rng.integers(
    low=0,
    high=NUM_GAMES,
    size=(
        EVALUATION_USERS,
        NEGATIVE_CANDIDATES,
    ),
    dtype=np.int64,
)

# Replace any random candidate that accidentally equals
# that user's held-out target game
target_matches = (
    random_candidate_indices
    == target_game_indices[:, None]
)

while target_matches.any():

    random_candidate_indices[target_matches] = rng.integers(
        low=0,
        high=NUM_GAMES,
        size=target_matches.sum(),
        dtype=np.int64,
    )

    target_matches = (
        random_candidate_indices
        == target_game_indices[:, None]
    )


print("Candidate array shape:", random_candidate_indices.shape)
print(
    "Candidates matching target:",
    (
        random_candidate_indices
        == target_game_indices[:, None]
    ).sum(),
)

Candidate array shape: (100000, 100)
Candidates matching target: 0


In [43]:
# ============================================================
# STEP 27: Score target games and random candidate games
# ============================================================

model.eval()

evaluation_user_indices = (
    positive_test_sample
    .get_column("user_idx")
    .to_numpy()
)

target_game_indices = (
    positive_test_sample
    .get_column("target_game_idx")
    .to_numpy()
)

# Convert target pairs to tensors
user_tensor = torch.tensor(
    evaluation_user_indices,
    dtype=torch.long,
    device=device,
)

target_game_tensor = torch.tensor(
    target_game_indices,
    dtype=torch.long,
    device=device,
)

with torch.no_grad():

    # Score each user's real held-out liked game
    target_scores = model(
        user_tensor,
        target_game_tensor,
    ).cpu().numpy()

    # Flatten users and candidates into one long batch
    repeated_user_indices = np.repeat(
        evaluation_user_indices,
        NEGATIVE_CANDIDATES,
    )

    flattened_candidate_indices = (
        random_candidate_indices.reshape(-1)
    )

    repeated_user_tensor = torch.tensor(
        repeated_user_indices,
        dtype=torch.long,
        device=device,
    )

    candidate_game_tensor = torch.tensor(
        flattened_candidate_indices,
        dtype=torch.long,
        device=device,
    )

    # Score all random candidate games
    candidate_scores = model(
        repeated_user_tensor,
        candidate_game_tensor,
    ).cpu().numpy()

# Restore candidate scores to:
# 100,000 users × 100 candidates
candidate_scores = candidate_scores.reshape(
    EVALUATION_USERS,
    NEGATIVE_CANDIDATES,
)

print("Target scores shape:", target_scores.shape)
print("Candidate scores shape:", candidate_scores.shape)

Target scores shape: (100000,)
Candidate scores shape: (100000, 100)


In [44]:
# ============================================================
# STEP 28: Calculate sampled recommendation ranking metrics
# ============================================================

# Rank 1 means the held-out liked game scored above
# all 100 random candidate games.
#
# Each candidate with a higher score pushes the target
# down by one position.
target_ranks = (
    1
    + (candidate_scores > target_scores[:, None]).sum(axis=1)
)

# Hit Rate@K:
# Percentage of users whose held-out liked game appears
# within the top K results.
hit_rate_at_1 = np.mean(target_ranks <= 1)
hit_rate_at_5 = np.mean(target_ranks <= 5)
hit_rate_at_10 = np.mean(target_ranks <= 10)

# Mean Reciprocal Rank:
# Gives more credit when the target is near the top.
mean_reciprocal_rank = np.mean(
    1.0 / target_ranks
)

median_rank = np.median(target_ranks)
mean_rank = np.mean(target_ranks)

print(f"Evaluation users: {len(target_ranks):,}")
print(f"Hit Rate@1:      {hit_rate_at_1:.4%}")
print(f"Hit Rate@5:      {hit_rate_at_5:.4%}")
print(f"Hit Rate@10:     {hit_rate_at_10:.4%}")
print(f"MRR:             {mean_reciprocal_rank:.4f}")
print(f"Median rank:     {median_rank:.1f}")
print(f"Mean rank:       {mean_rank:.2f}")

Evaluation users: 100,000
Hit Rate@1:      45.7870%
Hit Rate@5:      62.2090%
Hit Rate@10:     67.9330%
MRR:             0.5378
Median rank:     2.0
Mean rank:       19.65


In [45]:
# ============================================================
# STEP 29: Compare against a popularity-only baseline
# Uses the same users, targets, and random candidate games
# ============================================================

model.eval()

with torch.no_grad():

    # Extract one general score for every game.
    # Higher game bias means the game tends to receive
    # positive reviews regardless of which user reviewed it.
    game_bias_scores = (
        model.game_biases.weight
        .squeeze(dim=1)
        .detach()
        .cpu()
        .numpy()
    )

    global_bias_score = (
        model.global_bias
        .detach()
        .cpu()
        .item()
    )


# Score the held-out target games using popularity only
popularity_target_scores = (
    game_bias_scores[target_game_indices]
    + global_bias_score
)

# Score the same 100 candidate games using popularity only
popularity_candidate_scores = (
    game_bias_scores[random_candidate_indices]
    + global_bias_score
)

# Calculate each target game's rank
popularity_target_ranks = (
    1
    + (
        popularity_candidate_scores
        > popularity_target_scores[:, None]
    ).sum(axis=1)
)

# Calculate the same recommendation metrics
popularity_hit_rate_at_1 = np.mean(
    popularity_target_ranks <= 1
)

popularity_hit_rate_at_5 = np.mean(
    popularity_target_ranks <= 5
)

popularity_hit_rate_at_10 = np.mean(
    popularity_target_ranks <= 10
)

popularity_mrr = np.mean(
    1.0 / popularity_target_ranks
)

popularity_median_rank = np.median(
    popularity_target_ranks
)

popularity_mean_rank = np.mean(
    popularity_target_ranks
)

print("PERSONALIZED MODEL")
print(f"Hit Rate@1:  {hit_rate_at_1:.4%}")
print(f"Hit Rate@5:  {hit_rate_at_5:.4%}")
print(f"Hit Rate@10: {hit_rate_at_10:.4%}")
print(f"MRR:         {mean_reciprocal_rank:.4f}")
print(f"Median rank: {median_rank:.1f}")
print(f"Mean rank:   {mean_rank:.2f}")

print()

print("POPULARITY-ONLY BASELINE")
print(f"Hit Rate@1:  {popularity_hit_rate_at_1:.4%}")
print(f"Hit Rate@5:  {popularity_hit_rate_at_5:.4%}")
print(f"Hit Rate@10: {popularity_hit_rate_at_10:.4%}")
print(f"MRR:         {popularity_mrr:.4f}")
print(f"Median rank: {popularity_median_rank:.1f}")
print(f"Mean rank:   {popularity_mean_rank:.2f}")

PERSONALIZED MODEL
Hit Rate@1:  45.7870%
Hit Rate@5:  62.2090%
Hit Rate@10: 67.9330%
MRR:         0.5378
Median rank: 2.0
Mean rank:   19.65

POPULARITY-ONLY BASELINE
Hit Rate@1:  28.8440%
Hit Rate@5:  37.5550%
Hit Rate@10: 43.0060%
MRR:         0.3438
Median rank: 21.0
Mean rank:   34.90


In [46]:
# ============================================================
# STEP 30: Build a harder candidate pool from popular games
# ============================================================

POPULAR_GAME_POOL_SIZE = 5_000

popular_game_pool = (
    pl.scan_parquet("train_interactions.parquet")
    .group_by("game_idx")
    .agg(
        pl.len().alias("review_count")
    )
    .sort(
        "review_count",
        descending=True,
    )
    .head(POPULAR_GAME_POOL_SIZE)
    .select("game_idx")
    .collect()
    .get_column("game_idx")
    .to_numpy()
)

print("Popular candidate pool size:", len(popular_game_pool))
print("First few game indexes:", popular_game_pool[:10])

Popular candidate pool size: 5000
First few game indexes: [   29  4581  1887  8622 18556   161 37562    23 43276  3883]


In [47]:
# ============================================================
# STEP 31: Generate harder candidates from popular games
# ============================================================

popular_candidate_indices = rng.choice(
    popular_game_pool,
    size=(
        EVALUATION_USERS,
        NEGATIVE_CANDIDATES,
    ),
    replace=True,
)

# Replace any candidate that equals the user's target game
target_matches = (
    popular_candidate_indices
    == target_game_indices[:, None]
)

while target_matches.any():

    popular_candidate_indices[target_matches] = rng.choice(
        popular_game_pool,
        size=target_matches.sum(),
        replace=True,
    )

    target_matches = (
        popular_candidate_indices
        == target_game_indices[:, None]
    )


print(
    "Popular candidate array shape:",
    popular_candidate_indices.shape,
)

print(
    "Candidates matching target:",
    (
        popular_candidate_indices
        == target_game_indices[:, None]
    ).sum(),
)

Popular candidate array shape: (100000, 100)
Candidates matching target: 0


In [48]:
# ============================================================
# STEP 32: Evaluate against popular-game candidates
# ============================================================

# Flatten users and popular candidates for model scoring
repeated_user_indices = np.repeat(
    evaluation_user_indices,
    NEGATIVE_CANDIDATES,
)

flattened_popular_candidates = (
    popular_candidate_indices.reshape(-1)
)

repeated_user_tensor = torch.tensor(
    repeated_user_indices,
    dtype=torch.long,
    device=device,
)

popular_candidate_tensor = torch.tensor(
    flattened_popular_candidates,
    dtype=torch.long,
    device=device,
)

model.eval()

with torch.no_grad():

    popular_candidate_scores = model(
        repeated_user_tensor,
        popular_candidate_tensor,
    ).cpu().numpy()

popular_candidate_scores = popular_candidate_scores.reshape(
    EVALUATION_USERS,
    NEGATIVE_CANDIDATES,
)

# Personalized model ranks
popular_target_ranks = (
    1
    + (
        popular_candidate_scores
        > target_scores[:, None]
    ).sum(axis=1)
)

personalized_popular_hit_1 = np.mean(
    popular_target_ranks <= 1
)

personalized_popular_hit_5 = np.mean(
    popular_target_ranks <= 5
)

personalized_popular_hit_10 = np.mean(
    popular_target_ranks <= 10
)

personalized_popular_mrr = np.mean(
    1.0 / popular_target_ranks
)

personalized_popular_median_rank = np.median(
    popular_target_ranks
)

personalized_popular_mean_rank = np.mean(
    popular_target_ranks
)


# Popularity-only ranks using the same candidate set
popularity_popular_candidate_scores = (
    game_bias_scores[popular_candidate_indices]
    + global_bias_score
)

popularity_popular_target_ranks = (
    1
    + (
        popularity_popular_candidate_scores
        > popularity_target_scores[:, None]
    ).sum(axis=1)
)

baseline_popular_hit_1 = np.mean(
    popularity_popular_target_ranks <= 1
)

baseline_popular_hit_5 = np.mean(
    popularity_popular_target_ranks <= 5
)

baseline_popular_hit_10 = np.mean(
    popularity_popular_target_ranks <= 10
)

baseline_popular_mrr = np.mean(
    1.0 / popularity_popular_target_ranks
)

baseline_popular_median_rank = np.median(
    popularity_popular_target_ranks
)

baseline_popular_mean_rank = np.mean(
    popularity_popular_target_ranks
)


print("PERSONALIZED MODEL — POPULAR CANDIDATES")
print(f"Hit Rate@1:  {personalized_popular_hit_1:.4%}")
print(f"Hit Rate@5:  {personalized_popular_hit_5:.4%}")
print(f"Hit Rate@10: {personalized_popular_hit_10:.4%}")
print(f"MRR:         {personalized_popular_mrr:.4f}")
print(f"Median rank: {personalized_popular_median_rank:.1f}")
print(f"Mean rank:   {personalized_popular_mean_rank:.2f}")

print()

print("POPULARITY-ONLY — POPULAR CANDIDATES")
print(f"Hit Rate@1:  {baseline_popular_hit_1:.4%}")
print(f"Hit Rate@5:  {baseline_popular_hit_5:.4%}")
print(f"Hit Rate@10: {baseline_popular_hit_10:.4%}")
print(f"MRR:         {baseline_popular_mrr:.4f}")
print(f"Median rank: {baseline_popular_median_rank:.1f}")
print(f"Mean rank:   {baseline_popular_mean_rank:.2f}")

PERSONALIZED MODEL — POPULAR CANDIDATES
Hit Rate@1:  24.3140%
Hit Rate@5:  42.7790%
Hit Rate@10: 50.3790%
MRR:         0.3368
Median rank: 10.0
Mean rank:   28.06

POPULARITY-ONLY — POPULAR CANDIDATES
Hit Rate@1:  16.2350%
Hit Rate@5:  30.4890%
Hit Rate@10: 37.2880%
MRR:         0.2393
Median rank: 29.0
Mean rank:   41.01


In [49]:
# ============================================================
# STEP 33: Assign every game to a popularity bucket
# Games in the same bucket have similar review counts
# ============================================================

POPULARITY_BUCKETS = 20

game_popularity = (
    pl.scan_parquet("train_interactions.parquet")
    .group_by("game_idx")
    .agg(
        pl.len().alias("review_count")
    )
    .collect()
    .sort("review_count")
    .with_columns(
        (
            pl.col("review_count")
            .rank(method="ordinal")
            * POPULARITY_BUCKETS
            / pl.len()
        )
        .floor()
        .clip(
            lower_bound=0,
            upper_bound=POPULARITY_BUCKETS - 1,
        )
        .cast(pl.Int32)
        .alias("popularity_bucket")
    )
)

print("Games with popularity data:", game_popularity.height)
print(
    "Popularity buckets:",
    game_popularity
    .get_column("popularity_bucket")
    .n_unique(),
)

display(
    game_popularity
    .group_by("popularity_bucket")
    .agg(
        pl.len().alias("games"),
        pl.col("review_count").min().alias("minimum_reviews"),
        pl.col("review_count").median().alias("median_reviews"),
        pl.col("review_count").max().alias("maximum_reviews"),
    )
    .sort("popularity_bucket")
)

Games with popularity data: 74814
Popularity buckets: 20


popularity_bucket,games,minimum_reviews,median_reviews,maximum_reviews
i32,u32,u32,f64,u32
0,3740,1,5.0,5
1,3741,5,6.0,6
2,3741,6,7.0,7
3,3740,7,8.0,9
4,3741,9,10.0,11
…,…,…,…,…
15,3741,122,149.0,187
16,3740,187,239.0,318
17,3741,318,434.0,630


In [50]:
# ============================================================
# STEP 34: Generate popularity-matched candidate games
# ============================================================

# Create a lookup:
# game_idx -> popularity bucket
game_to_bucket = np.full(
    NUM_GAMES,
    fill_value=-1,
    dtype=np.int32,
)

game_to_bucket[
    game_popularity.get_column("game_idx").to_numpy()
] = (
    game_popularity
    .get_column("popularity_bucket")
    .to_numpy()
)

# Create one array of game indexes for each bucket
games_by_bucket = {}

for bucket_number in range(POPULARITY_BUCKETS):

    games_by_bucket[bucket_number] = (
        game_popularity
        .filter(
            pl.col("popularity_bucket") == bucket_number
        )
        .get_column("game_idx")
        .to_numpy()
    )


# Find the popularity bucket for every target game
target_popularity_buckets = game_to_bucket[
    target_game_indices
]

if (target_popularity_buckets < 0).any():
    raise ValueError(
        "At least one target game has no popularity bucket."
    )


# Allocate the candidate array
matched_candidate_indices = np.empty(
    (
        EVALUATION_USERS,
        NEGATIVE_CANDIDATES,
    ),
    dtype=np.int64,
)


# Sample candidates bucket by bucket
for bucket_number in range(POPULARITY_BUCKETS):

    user_mask = (
        target_popularity_buckets == bucket_number
    )

    user_positions = np.flatnonzero(user_mask)

    if len(user_positions) == 0:
        continue

    bucket_games = games_by_bucket[bucket_number]

    matched_candidate_indices[user_positions] = rng.choice(
        bucket_games,
        size=(
            len(user_positions),
            NEGATIVE_CANDIDATES,
        ),
        replace=True,
    )


# Replace any candidate equal to that user's target game
target_matches = (
    matched_candidate_indices
    == target_game_indices[:, None]
)

while target_matches.any():

    matching_rows, matching_columns = np.where(
        target_matches
    )

    for bucket_number in range(POPULARITY_BUCKETS):

        bucket_match_mask = (
            target_popularity_buckets[matching_rows]
            == bucket_number
        )

        rows_for_bucket = matching_rows[bucket_match_mask]
        columns_for_bucket = matching_columns[
            bucket_match_mask
        ]

        if len(rows_for_bucket) == 0:
            continue

        matched_candidate_indices[
            rows_for_bucket,
            columns_for_bucket,
        ] = rng.choice(
            games_by_bucket[bucket_number],
            size=len(rows_for_bucket),
            replace=True,
        )

    target_matches = (
        matched_candidate_indices
        == target_game_indices[:, None]
    )


print(
    "Matched candidate array shape:",
    matched_candidate_indices.shape,
)

print(
    "Candidates matching target:",
    (
        matched_candidate_indices
        == target_game_indices[:, None]
    ).sum(),
)

print(
    "Users missing a target bucket:",
    (target_popularity_buckets < 0).sum(),
)

ValueError: At least one target game has no popularity bucket.

In [52]:
# ============================================================
# STEP 34A: Keep only targets that exist in training popularity
# ============================================================

target_popularity_buckets = game_to_bucket[
    target_game_indices
]

valid_target_mask = target_popularity_buckets >= 0

print(
    "Targets present in training:",
    valid_target_mask.sum(),
)

print(
    "Targets missing from training:",
    (~valid_target_mask).sum(),
)


# Filter every evaluation array to the same valid users
evaluation_user_indices = evaluation_user_indices[
    valid_target_mask
]

target_game_indices = target_game_indices[
    valid_target_mask
]

target_scores = target_scores[
    valid_target_mask
]

target_popularity_buckets = target_popularity_buckets[
    valid_target_mask
]

# Update the evaluation-user count
EVALUATION_USERS = len(target_game_indices)

print(
    "Updated evaluation users:",
    EVALUATION_USERS,
)

Targets present in training: 99999
Targets missing from training: 0
Updated evaluation users: 99999


In [53]:
# ============================================================
# STEP 34B: Generate popularity-matched candidate games
# ============================================================

matched_candidate_indices = np.empty(
    (
        EVALUATION_USERS,
        NEGATIVE_CANDIDATES,
    ),
    dtype=np.int64,
)

for bucket_number in range(POPULARITY_BUCKETS):

    user_positions = np.flatnonzero(
        target_popularity_buckets == bucket_number
    )

    if len(user_positions) == 0:
        continue

    bucket_games = games_by_bucket[bucket_number]

    matched_candidate_indices[user_positions] = rng.choice(
        bucket_games,
        size=(
            len(user_positions),
            NEGATIVE_CANDIDATES,
        ),
        replace=True,
    )


# Replace candidates equal to the held-out target
target_matches = (
    matched_candidate_indices
    == target_game_indices[:, None]
)

while target_matches.any():

    matching_rows, matching_columns = np.where(
        target_matches
    )

    for bucket_number in range(POPULARITY_BUCKETS):

        bucket_mask = (
            target_popularity_buckets[matching_rows]
            == bucket_number
        )

        rows_for_bucket = matching_rows[bucket_mask]
        columns_for_bucket = matching_columns[bucket_mask]

        if len(rows_for_bucket) == 0:
            continue

        matched_candidate_indices[
            rows_for_bucket,
            columns_for_bucket,
        ] = rng.choice(
            games_by_bucket[bucket_number],
            size=len(rows_for_bucket),
            replace=True,
        )

    target_matches = (
        matched_candidate_indices
        == target_game_indices[:, None]
    )


print(
    "Matched candidate array shape:",
    matched_candidate_indices.shape,
)

print(
    "Candidates matching target:",
    (
        matched_candidate_indices
        == target_game_indices[:, None]
    ).sum(),
)

Matched candidate array shape: (99999, 100)
Candidates matching target: 0


In [54]:
# ============================================================
# STEP 35: Evaluate popularity-matched candidates
# ============================================================

repeated_user_indices = np.repeat(
    evaluation_user_indices,
    NEGATIVE_CANDIDATES,
)

flattened_matched_candidates = (
    matched_candidate_indices.reshape(-1)
)

repeated_user_tensor = torch.tensor(
    repeated_user_indices,
    dtype=torch.long,
    device=device,
)

matched_candidate_tensor = torch.tensor(
    flattened_matched_candidates,
    dtype=torch.long,
    device=device,
)

model.eval()

with torch.no_grad():

    matched_candidate_scores = model(
        repeated_user_tensor,
        matched_candidate_tensor,
    ).cpu().numpy()

matched_candidate_scores = matched_candidate_scores.reshape(
    EVALUATION_USERS,
    NEGATIVE_CANDIDATES,
)


# Personalized-model ranks
matched_target_ranks = (
    1
    + (
        matched_candidate_scores
        > target_scores[:, None]
    ).sum(axis=1)
)

personalized_matched_hit_1 = np.mean(
    matched_target_ranks <= 1
)

personalized_matched_hit_5 = np.mean(
    matched_target_ranks <= 5
)

personalized_matched_hit_10 = np.mean(
    matched_target_ranks <= 10
)

personalized_matched_mrr = np.mean(
    1.0 / matched_target_ranks
)

personalized_matched_median_rank = np.median(
    matched_target_ranks
)

personalized_matched_mean_rank = np.mean(
    matched_target_ranks
)


# Popularity-only ranks on the same candidates
matched_popularity_target_scores = (
    game_bias_scores[target_game_indices]
    + global_bias_score
)

matched_popularity_candidate_scores = (
    game_bias_scores[matched_candidate_indices]
    + global_bias_score
)

matched_popularity_ranks = (
    1
    + (
        matched_popularity_candidate_scores
        > matched_popularity_target_scores[:, None]
    ).sum(axis=1)
)

baseline_matched_hit_1 = np.mean(
    matched_popularity_ranks <= 1
)

baseline_matched_hit_5 = np.mean(
    matched_popularity_ranks <= 5
)

baseline_matched_hit_10 = np.mean(
    matched_popularity_ranks <= 10
)

baseline_matched_mrr = np.mean(
    1.0 / matched_popularity_ranks
)

baseline_matched_median_rank = np.median(
    matched_popularity_ranks
)

baseline_matched_mean_rank = np.mean(
    matched_popularity_ranks
)


print("PERSONALIZED MODEL — POPULARITY MATCHED")
print(f"Hit Rate@1:  {personalized_matched_hit_1:.4%}")
print(f"Hit Rate@5:  {personalized_matched_hit_5:.4%}")
print(f"Hit Rate@10: {personalized_matched_hit_10:.4%}")
print(f"MRR:         {personalized_matched_mrr:.4f}")
print(f"Median rank: {personalized_matched_median_rank:.1f}")
print(f"Mean rank:   {personalized_matched_mean_rank:.2f}")

print()

print("POPULARITY-ONLY — POPULARITY MATCHED")
print(f"Hit Rate@1:  {baseline_matched_hit_1:.4%}")
print(f"Hit Rate@5:  {baseline_matched_hit_5:.4%}")
print(f"Hit Rate@10: {baseline_matched_hit_10:.4%}")
print(f"MRR:         {baseline_matched_mrr:.4f}")
print(f"Median rank: {baseline_matched_median_rank:.1f}")
print(f"Mean rank:   {baseline_matched_mean_rank:.2f}")

PERSONALIZED MODEL — POPULARITY MATCHED
Hit Rate@1:  22.0662%
Hit Rate@5:  40.9564%
Hit Rate@10: 49.0755%
MRR:         0.3169
Median rank: 11.0
Mean rank:   28.30

POPULARITY-ONLY — POPULARITY MATCHED
Hit Rate@1:  14.6791%
Hit Rate@5:  28.9753%
Hit Rate@10: 36.2824%
MRR:         0.2245
Median rank: 30.0
Mean rank:   41.44


In [55]:
# ============================================================
# STEP 36: Audit the popularity-matched candidate set
# Check duplicate candidates and previously reviewed games
# ============================================================

# ------------------------------------------------------------
# 1. Measure duplicate candidates within each user's 100 games
# ------------------------------------------------------------

unique_candidates_per_user = np.array(
    [
        np.unique(row).size
        for row in matched_candidate_indices
    ]
)

print("CANDIDATE DUPLICATES")
print(
    "Average unique candidates per user:",
    unique_candidates_per_user.mean(),
)
print(
    "Users with at least one duplicate:",
    np.mean(
        unique_candidates_per_user < NEGATIVE_CANDIDATES
    ),
)
print(
    "Minimum unique candidates for one user:",
    unique_candidates_per_user.min(),
)


# ------------------------------------------------------------
# 2. Build a user-candidate table for overlap checking
# ------------------------------------------------------------

candidate_pairs = pl.DataFrame(
    {
        "evaluation_row": np.repeat(
            np.arange(EVALUATION_USERS),
            NEGATIVE_CANDIDATES,
        ),
        "user_idx": np.repeat(
            evaluation_user_indices,
            NEGATIVE_CANDIDATES,
        ),
        "game_idx": matched_candidate_indices.reshape(-1),
    }
)

# Historical user-game pairs from the training data
reviewed_pairs = (
    pl.scan_parquet("train_interactions.parquet")
    .select(
        "user_idx",
        "game_idx",
    )
    .unique()
)

candidate_audit = (
    candidate_pairs.lazy()
    .join(
        reviewed_pairs.with_columns(
            pl.lit(True).alias("already_reviewed")
        ),
        on=[
            "user_idx",
            "game_idx",
        ],
        how="left",
    )
    .with_columns(
        pl.col("already_reviewed")
        .fill_null(False)
    )
    .select(
        pl.len().alias("candidate_rows"),
        pl.col("already_reviewed")
        .sum()
        .alias("already_reviewed_rows"),
        pl.col("already_reviewed")
        .mean()
        .alias("already_reviewed_rate"),
    )
    .collect()
)

print()
print("TRAINING-HISTORY OVERLAP")
display(candidate_audit)

CANDIDATE DUPLICATES
Average unique candidates per user: 98.68375683756838
Users with at least one duplicate: 0.7373673736737367
Minimum unique candidates for one user: 91

TRAINING-HISTORY OVERLAP


candidate_rows,already_reviewed_rows,already_reviewed_rate
u32,u32,f64
9999900,14022,0.001402


In [56]:
# ============================================================
# STEP 37: Generate unique popularity-matched candidates
# Samples without replacement within each user's candidate set
# ============================================================

unique_matched_candidate_indices = np.empty(
    (
        EVALUATION_USERS,
        NEGATIVE_CANDIDATES,
    ),
    dtype=np.int64,
)

for bucket_number in range(POPULARITY_BUCKETS):

    user_positions = np.flatnonzero(
        target_popularity_buckets == bucket_number
    )

    if len(user_positions) == 0:
        continue

    bucket_games = games_by_bucket[bucket_number]

    if len(bucket_games) <= NEGATIVE_CANDIDATES:
        raise ValueError(
            f"Bucket {bucket_number} has only "
            f"{len(bucket_games)} games."
        )

    for user_position in user_positions:

        target_game = target_game_indices[user_position]

        # Remove the user's held-out target from the pool
        eligible_games = bucket_games[
            bucket_games != target_game
        ]

        unique_matched_candidate_indices[
            user_position
        ] = rng.choice(
            eligible_games,
            size=NEGATIVE_CANDIDATES,
            replace=False,
        )


# Verify that every row contains 100 distinct games
unique_counts = np.array(
    [
        np.unique(row).size
        for row in unique_matched_candidate_indices
    ]
)

print(
    "Unique candidate array shape:",
    unique_matched_candidate_indices.shape,
)

print(
    "Candidates matching target:",
    (
        unique_matched_candidate_indices
        == target_game_indices[:, None]
    ).sum(),
)

print(
    "Minimum unique candidates per user:",
    unique_counts.min(),
)

print(
    "Average unique candidates per user:",
    unique_counts.mean(),
)

Unique candidate array shape: (99999, 100)
Candidates matching target: 0
Minimum unique candidates per user: 100
Average unique candidates per user: 100.0


In [57]:
# ============================================================
# STEP 38: Evaluate unique popularity-matched candidates
# ============================================================

repeated_user_indices = np.repeat(
    evaluation_user_indices,
    NEGATIVE_CANDIDATES,
)

flattened_unique_candidates = (
    unique_matched_candidate_indices.reshape(-1)
)

repeated_user_tensor = torch.tensor(
    repeated_user_indices,
    dtype=torch.long,
    device=device,
)

unique_candidate_tensor = torch.tensor(
    flattened_unique_candidates,
    dtype=torch.long,
    device=device,
)

model.eval()

with torch.no_grad():

    unique_candidate_scores = model(
        repeated_user_tensor,
        unique_candidate_tensor,
    ).cpu().numpy()

unique_candidate_scores = unique_candidate_scores.reshape(
    EVALUATION_USERS,
    NEGATIVE_CANDIDATES,
)


# Personalized-model ranks
unique_target_ranks = (
    1
    + (
        unique_candidate_scores
        > target_scores[:, None]
    ).sum(axis=1)
)

personalized_unique_hit_1 = np.mean(
    unique_target_ranks <= 1
)

personalized_unique_hit_5 = np.mean(
    unique_target_ranks <= 5
)

personalized_unique_hit_10 = np.mean(
    unique_target_ranks <= 10
)

personalized_unique_mrr = np.mean(
    1.0 / unique_target_ranks
)

personalized_unique_median_rank = np.median(
    unique_target_ranks
)

personalized_unique_mean_rank = np.mean(
    unique_target_ranks
)


# Popularity-only ranks on the exact same candidate set
unique_popularity_target_scores = (
    game_bias_scores[target_game_indices]
    + global_bias_score
)

unique_popularity_candidate_scores = (
    game_bias_scores[unique_matched_candidate_indices]
    + global_bias_score
)

unique_popularity_ranks = (
    1
    + (
        unique_popularity_candidate_scores
        > unique_popularity_target_scores[:, None]
    ).sum(axis=1)
)

baseline_unique_hit_1 = np.mean(
    unique_popularity_ranks <= 1
)

baseline_unique_hit_5 = np.mean(
    unique_popularity_ranks <= 5
)

baseline_unique_hit_10 = np.mean(
    unique_popularity_ranks <= 10
)

baseline_unique_mrr = np.mean(
    1.0 / unique_popularity_ranks
)

baseline_unique_median_rank = np.median(
    unique_popularity_ranks
)

baseline_unique_mean_rank = np.mean(
    unique_popularity_ranks
)


print("PERSONALIZED MODEL — UNIQUE POPULARITY MATCHED")
print(f"Hit Rate@1:  {personalized_unique_hit_1:.4%}")
print(f"Hit Rate@5:  {personalized_unique_hit_5:.4%}")
print(f"Hit Rate@10: {personalized_unique_hit_10:.4%}")
print(f"MRR:         {personalized_unique_mrr:.4f}")
print(f"Median rank: {personalized_unique_median_rank:.1f}")
print(f"Mean rank:   {personalized_unique_mean_rank:.2f}")

print()

print("POPULARITY-ONLY — UNIQUE POPULARITY MATCHED")
print(f"Hit Rate@1:  {baseline_unique_hit_1:.4%}")
print(f"Hit Rate@5:  {baseline_unique_hit_5:.4%}")
print(f"Hit Rate@10: {baseline_unique_hit_10:.4%}")
print(f"MRR:         {baseline_unique_mrr:.4f}")
print(f"Median rank: {baseline_unique_median_rank:.1f}")
print(f"Mean rank:   {baseline_unique_mean_rank:.2f}")

PERSONALIZED MODEL — UNIQUE POPULARITY MATCHED
Hit Rate@1:  22.0062%
Hit Rate@5:  40.9234%
Hit Rate@10: 48.9995%
MRR:         0.3162
Median rank: 11.0
Mean rank:   28.31

POPULARITY-ONLY — UNIQUE POPULARITY MATCHED
Hit Rate@1:  14.5001%
Hit Rate@5:  28.8593%
Hit Rate@10: 36.1994%
MRR:         0.2234
Median rank: 30.0
Mean rank:   41.42


In [58]:
# ============================================================
# STEP 39: Save the primary offline evaluation results
# ============================================================

evaluation_results = pl.DataFrame(
    {
        "model": [
            "personalized_matrix_factorization",
            "popularity_only",
        ],
        "evaluation": [
            "unique_popularity_matched_100_candidates",
            "unique_popularity_matched_100_candidates",
        ],
        "evaluation_users": [
            EVALUATION_USERS,
            EVALUATION_USERS,
        ],
        "hit_rate_at_1": [
            personalized_unique_hit_1,
            baseline_unique_hit_1,
        ],
        "hit_rate_at_5": [
            personalized_unique_hit_5,
            baseline_unique_hit_5,
        ],
        "hit_rate_at_10": [
            personalized_unique_hit_10,
            baseline_unique_hit_10,
        ],
        "mrr": [
            personalized_unique_mrr,
            baseline_unique_mrr,
        ],
        "median_rank": [
            personalized_unique_median_rank,
            baseline_unique_median_rank,
        ],
        "mean_rank": [
            personalized_unique_mean_rank,
            baseline_unique_mean_rank,
        ],
    }
)

evaluation_results.write_parquet(
    "matrix_factorization_evaluation.parquet",
    compression="zstd",
)

display(evaluation_results)

print("Saved matrix_factorization_evaluation.parquet")

model,evaluation,evaluation_users,hit_rate_at_1,hit_rate_at_5,hit_rate_at_10,mrr,median_rank,mean_rank
str,str,i64,f64,f64,f64,f64,f64,f64
"""personalized_matrix_factorizat…","""unique_popularity_matched_100_…",99999,0.220062,0.409234,0.489995,0.316201,11.0,28.314903
"""popularity_only""","""unique_popularity_matched_100_…",99999,0.145001,0.288593,0.361994,0.223432,30.0,41.421344


Saved matrix_factorization_evaluation.parquet


In [59]:
# ============================================================
# STEP 40: Build a static recommendation table for every game
# Uses cosine similarity between learned game embeddings
# ============================================================

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F


RECOMMENDATIONS_PER_GAME = 30
SIMILARITY_BATCH_SIZE = 1_024


# ------------------------------------------------------------
# 1. Extract and normalize the learned game embeddings
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    game_embedding_matrix = (
        model.game_embeddings.weight
        .detach()
        .to(device)
    )

    # Normalization makes the dot product equal cosine similarity
    normalized_game_embeddings = F.normalize(
        game_embedding_matrix,
        p=2,
        dim=1,
    )


# ------------------------------------------------------------
# 2. Find the most similar games in manageable GPU batches
# ------------------------------------------------------------

all_recommendation_indices = []

for start_idx in range(
    0,
    NUM_GAMES,
    SIMILARITY_BATCH_SIZE,
):

    end_idx = min(
        start_idx + SIMILARITY_BATCH_SIZE,
        NUM_GAMES,
    )

    batch_embeddings = normalized_game_embeddings[
        start_idx:end_idx
    ]

    # Compare this batch against every game
    similarity_scores = (
        batch_embeddings
        @ normalized_game_embeddings.T
    )

    # Prevent each game from recommending itself
    batch_row_indices = torch.arange(
        end_idx - start_idx,
        device=device,
    )

    batch_game_indices = torch.arange(
        start_idx,
        end_idx,
        device=device,
    )

    similarity_scores[
        batch_row_indices,
        batch_game_indices,
    ] = -torch.inf

    # Keep only the 30 nearest games
    recommendation_indices = torch.topk(
        similarity_scores,
        k=RECOMMENDATIONS_PER_GAME,
        dim=1,
    ).indices

    all_recommendation_indices.append(
        recommendation_indices.cpu().numpy()
    )

    if start_idx % 10_240 == 0:
        print(
            f"Processed games "
            f"{start_idx:,} through {end_idx - 1:,}"
        )


all_recommendation_indices = np.vstack(
    all_recommendation_indices
)

print(
    "Recommendation index array shape:",
    all_recommendation_indices.shape,
)

Processed games 0 through 1,023
Processed games 10,240 through 11,263
Processed games 20,480 through 21,503
Processed games 30,720 through 31,743
Processed games 40,960 through 41,983
Processed games 51,200 through 52,223
Processed games 61,440 through 62,463
Processed games 71,680 through 72,703
Recommendation index array shape: (74815, 30)


In [60]:
# ============================================================
# STEP 41: Convert recommendation indexes to app IDs
# and save the static recommendation table
# ============================================================

# Ensure game_index is ordered by game_idx
ordered_game_index = (
    game_index
    .sort("game_idx")
)

appid_lookup = (
    ordered_game_index
    .get_column("appid")
    .to_numpy()
)

# Convert each recommended game_idx into its Steam appid
recommendation_appids = appid_lookup[
    all_recommendation_indices
]

# Build one row per source game
recommendation_table = pl.DataFrame(
    {
        "appid": appid_lookup,
        "recommendations": recommendation_appids.tolist(),
    }
)

recommendation_table.write_parquet(
    "game_recommendations.parquet",
    compression="zstd",
)

print(
    "Saved game_recommendations.parquet"
)

print(
    "Rows:",
    recommendation_table.height,
)

display(
    recommendation_table.head()
)

Saved game_recommendations.parquet
Rows: 74815


appid,recommendations
i64,list[i64]
10,"[6020, 346170, … 1380390]"
20,"[30, 1907360, … 787070]"
30,"[240, 1604000, … 70617]"
40,"[1780970, 2200370, … 1769510]"
50,"[342300, 629730, … 4000]"
